In [14]:
import pandas as pd
import numpy as np

In [15]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Dwarka-Sector 8, Delhi - DPCC.xlsx",skiprows=16)

In [16]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [17]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [18]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [ ]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [19]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date    PM2.5    PM10      NO     NO2  \
0  01-01-2025 00:00  02-01-2025 00:00  160.560  238.11   43.94   53.85   
1  02-01-2025 00:00  03-01-2025 00:00  185.960  264.50   37.12   49.70   
2  03-01-2025 00:00  04-01-2025 00:00   65.955  376.33  110.37   75.11   
3  04-01-2025 00:00  05-01-2025 00:00   65.955  328.95   41.33  104.60   
4  05-01-2025 00:00  06-01-2025 00:00  181.880  254.17   16.90   62.07   

      NOx      NH3   SO2     CO  Ozone  Benzene  Toluene     RH    WS      WD  \
0   64.31  100.790  3.45  1.820  22.37     2.67     8.91  87.17  0.90  250.24   
1   56.74   40.505  5.55  2.200  16.24     3.10    11.62  88.24  0.95  255.40   
2  129.69   40.505  6.75  1.275  42.13     1.42     9.79  86.89  0.74  237.78   
3   88.87   40.505  2.70  1.275  74.64     1.42     9.79  85.64  1.08  165.32   
4   46.79   40.505  2.34  1.870  19.12     3.63    10.50  84.76  1.44  156.24   

       BP     AT   RF  TOT-RF  
0  986.00  13

In [20]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [21]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,1.972616,0.421634,0.506599,0.071051,0.310546,2.849771,-2.092549,0.763544,-1.145236,0.625543,-0.311155,1.567770,-0.188411,1.068980,0.826871,-2.293360,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.541320,0.694342,0.239348,-0.123678,0.069518,-0.159948,-1.573070,1.447499,-1.496711,0.923234,0.085986,1.643450,0.004955,1.188155,0.829652,-2.254790,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.145583,1.849966,3.109746,1.068625,2.392242,-0.159948,-1.276225,-0.217392,-0.012261,-0.239841,-0.182194,1.547966,-0.807184,0.781202,0.829652,-2.100512,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.145583,1.360353,0.404323,2.452372,1.092535,-0.159948,-2.278077,-0.217392,1.851759,-0.239841,-0.182194,1.459556,0.507708,-0.892339,0.868580,-1.979294,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.449969,0.587595,-0.552999,0.456755,-0.247290,-0.159948,-2.367131,0.853538,-1.331581,1.290157,-0.078146,1.397315,1.899946,-1.102052,0.842164,-2.054596,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.145583,-0.160983,1.955317,1.665481,1.844913,1.229710,0.255001,1.447499,0.092092,2.820155,-0.182194,-0.062522,-1.387283,0.771502,0.590523,-1.207903,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.145583,2.786101,0.811469,1.991592,1.207159,1.561710,0.428160,1.627488,-0.095973,2.314771,1.359474,0.218270,-1.657996,0.640778,0.817139,-1.290552,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.145583,2.068318,0.063402,1.795456,0.642000,1.339544,1.385486,1.555492,0.189564,2.411694,1.350681,0.032254,-1.619323,0.706833,0.755967,-1.279532,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.145583,2.157395,1.237816,1.770587,1.408387,1.564206,1.390433,1.717482,-0.005381,2.889386,1.878248,0.115714,-1.541976,0.798986,0.924191,-1.345652,0.0,0.0


In [22]:
df.to_excel('Dwarkasector82025.xlsx', index=False)